## 2.4 文档嵌入模型 Text Embedding Models
### 2.4.1 嵌入模型概述
Text Embedding Models：文档嵌入模型，提供将文本编码为向量的能力，即 `文档向量化` 。 文档写入 和 用户查询匹配 前都会先执行文档嵌入编码，即向量化。

嵌入模型把一段文本变成一个**固定长度的浮点数列表**（向量）：

* 列表有多长（**维度**）由模型决定，和文本长短无关。2.4.3 里同一句话，Qwen3-VL-Embedding-2B 给出 1024 个数，jina-embeddings-v4 给出 2048 个；
* 训练时让意思相近的文本得到相近的向量。所以“猫喜欢吃鱼”和“小猫爱吃鱼干”字面不同，向量却很接近，这就是向量检索能“按意思找”而不只是“按字找”的原因。

![嵌入模型在 RAG 中的位置](assets/11-embedding-in-rag.svg)

“相近”用相似度来衡量，官方文档列了三种：

| 度量 | 算的是什么 | 越相似时 |
|---|---|---|
| 余弦相似度（cosine） | 两个向量夹角的余弦，只看方向，不看长度 | 越大，最大为 1 |
| 欧氏距离（Euclidean） | 两个点之间的直线距离 | 越小 |
| 点积（dot product） | 对应位置相乘再相加，同时受方向和长度影响 | 越大 |

向量长度都是 1（**归一化**）时，三者给出的排序完全一样：点积就等于余弦相似度，欧氏距离的平方等于 2 − 2 × 余弦相似度。模型返回的向量是不是归一化的，各家不一样，2.4.3 有一个反例。

常用的嵌入模型（2026-10 查到的信息。新模型出得很快，选型前再看一下各家文档和 MTEB 榜单）：

**托管 API**（按调用量付费，数据要发到服务商）

| 模型 | 厂商 | 默认维度（可选） | 最大输入 | 说明 |
|---|---|---|---|---|
| `text-embedding-3-small` | OpenAI | 1536（`dimensions` 可缩短） | 8191 token | 便宜，官方文档推荐做原型 |
| `text-embedding-3-large` | OpenAI | 3072（可缩短） | 8191 token | OpenAI 目前效果最好的嵌入模型 |
| `text-embedding-ada-002` | OpenAI | 1536（固定） | 8191 token | 2022 年的旧模型，老教程常见。`OpenAIEmbeddings()` 不传 `model` 时默认用的还是它 |
| `gemini-embedding-001` | Google | 3072（推荐 768 / 1536 / 3072） | 2048 token | |
| `gemini-embedding-2-preview` | Google | 3072（可缩短） | | 2026-03 发布的预览版，能嵌入图片等；向量和 001 不通用 |
| `voyage-4-large` / `voyage-4` / `voyage-4-lite` | Voyage AI | 1024（256 / 512 / 2048） | 32K token | 三个模型共用一个向量空间；区分查询和文档（`input_type`） |
| `embed-v4.0` | Cohere | 1536（256 / 512 / 1024） | 128K token | 能嵌入图片；区分查询和文档 |
| ⭐ `text-embedding-v4` | 阿里云百炼 | 1024（64～2048） | 8192 token | 基于 Qwen3-Embedding，中文效果好；一次请求最多 10 条 |

**开源模型**（可以下载到本地运行，没有调用费，数据不出本机）

| 模型 | 发布方 | 维度 | 最大输入 | 说明 |
|---|---|---|---|---|
| ⭐ `Qwen3-Embedding-0.6B` / `4B` / `8B` | 阿里 Qwen | 1024 / 2560 / 4096（可缩短） | 32K | 多语言，中文好；查询要加一句任务指令，文档不加（2.4.3） |
| `Qwen3-VL-Embedding-2B` | 阿里 Qwen | 2048（64～2048） | 32K | 2026-01 发布，文本、图片、视频都能嵌入。本笔记的 `embedding_model` |
| ⭐ `BAAI/bge-m3` | 智源 BAAI | 1024 | 8192 | 一次同时输出稠密、稀疏、多向量三种表示；中文项目常用 |
| `BAAI/bge-large-zh-v1.5` | 智源 BAAI | 1024 | 512 | 老牌中文模型，查询前可以加指令，v1.5 不加只是略差（2.4.3） |
| `jina-embeddings-v4` | Jina AI | 2048（128～2048） | 32K | 基于 Qwen2.5-VL-3B，能嵌入图片。本笔记的 `embedding_model2`。许可证是 Qwen Research License，商用前要看授权 |
| `jina-embeddings-v5-text-small` / `nano` | Jina AI | 1024 / 768 | 32K / 8K | 2026-02 发布的小模型，只支持文本 |
| `intfloat/multilingual-e5-large` | 微软 | 1024 | 512 | 查询加 `query: `，文档加 `passage: ` |
| `google/embeddinggemma-300m` | Google | 768（可缩短） | 2048 | 3 亿参数，CPU 上也能跑 |
| `sentence-transformers/all-MiniLM-L6-v2` | SBERT | 384 | 256 | 很小很快，只支持英文，常用来做演示 |

⭐ 是中文项目可以先试的。

**在 LangChain 里怎么调用**

| 部署方式 | LangChain 类 | 安装的包 | `init_embeddings` 前缀（2.4.2） |
|---|---|---|---|
| ⭐ OpenAI，以及任何 OpenAI 兼容接口（中转站、百炼兼容模式、vLLM、TEI 等） | `OpenAIEmbeddings` | `langchain-openai` | `openai` |
| Google Gemini API | `GoogleGenerativeAIEmbeddings` | `langchain-google-genai` | `google_genai` |
| Cohere | `CohereEmbeddings` | `langchain-cohere` | `cohere` |
| Voyage AI | `VoyageAIEmbeddings` | `langchain-voyageai` | 没有，只能直接用类 |
| ⭐ 本地运行 Hugging Face 上的模型 | `HuggingFaceEmbeddings` | `langchain-huggingface`（还要装 PyTorch） | `huggingface` |
| 本地 Ollama 服务 | `OllamaEmbeddings` | `langchain-ollama` | `ollama` |

学习重点是 `OpenAIEmbeddings`：国内的百炼、智谱，自己部署的 vLLM、TEI，还有本笔记用的中转站，都提供 OpenAI 兼容接口，一个类就能调用。

> ⚠️ **与旧教程不同**：老教程常用 `langchain_community` 里的 `DashScopeEmbeddings`（阿里）、`QianfanEmbeddingsEndpoint`（百度）、`HuggingFaceEmbeddings` 等类。`langchain-community` 已在 2026-05-22 停止维护；`langchain` 1.0 也把社区的嵌入类都移到了 `langchain-classic`（见 `langchain/embeddings/__init__.py` 的说明）。现在国内服务用 `OpenAIEmbeddings` 加 `base_url` 调用，本地模型用 `langchain-huggingface` 的 `HuggingFaceEmbeddings`。

> ⚠️ **官方文档的说明**：1.x 的 Embedding models 页写明，LangChain 目前只支持**文本**嵌入，不支持多模态嵌入。`Embeddings` 接口的两个方法都只收字符串，所以 Qwen3-VL-Embedding、jina-embeddings-v4 嵌入图片的能力，在 LangChain 里用不上。

> ⚠️ **官方文档有误**：官方页 Dimensionality 一节把 `Qwen3-Embedding-0.6B` 写成 1536 维，和 `text-embedding-3-small` 放在一档；它的模型卡写的是 1024 维。另外官方页举例的 `voyage-3`、Cohere `embed-english-v3` 都已经不是各家的最新模型（Voyage 文档把 3 系列列在 Older models 里）。

LangChain中针对向量化模型的封装提供了两种接口，一种针对 查询的向量化 `embed_query` ，一种针对 文档的向量化 `embed_documents` 。

> ⚠️ **与官方文档不同**：原笔记把 `embed_query` 叫“句子的向量化”。v0.1 和 1.x 的官方文档都说它是给**查询**（用户的搜索问题）用的，传一整段话也可以，和“句子”无关。分成两个方法，是因为有些模型对查询和文档的处理方式不同（2.4.3）。

两个方法定义在所有嵌入类的父类 `Embeddings` 里（`langchain_core/embeddings/embeddings.py`）：

| 方法 | 参数 | 返回值 | 说明 |
|---|---|---|---|
| `embed_documents(texts)` | `list[str]` | `list[list[float]]`，每条文本一个向量 | 抽象方法，子类必须实现 |
| `embed_query(text)` | `str` | `list[float]`，一个向量 | 抽象方法，子类必须实现 |
| `aembed_documents` / `aembed_query` | 同上 | 同上 | 异步版本。父类默认把同步方法放到线程池里执行（`run_in_executor`），子类可以重写成真正的异步请求 |

源码注释说：“通常查询的向量和文档的向量是一样的，但这个抽象允许分开处理。”1.x 文档也提到，多数服务实际上对两者的处理方式相同。

### 2.4.2 嵌入模型的选择 

选嵌入模型主要看这几点（参考官方文档的 Factors to weigh）：

| 因素 | 要看什么 |
|---|---|
| 效果 | 先看 [MTEB 榜单](https://huggingface.co/spaces/mteb/leaderboard)，按语言（中文）和任务（Retrieval，检索）筛选。榜单分数不一定适用于你的数据，最后用自己的几十个问题测一测 |
| 语言 | 中文项目选在中文上训练过的：Qwen3-Embedding、bge-m3、text-embedding-v4 等。只用英文训练的模型分不清中文的意思（2.3.4.4 的 potion-base-32M 就是例子） |
| 最大输入长度 | 要大于切分出来的块长（2.3）。老模型常只有 512 token，超出的部分一般被截掉，不参与向量 |
| 维度 | 维度越高，向量库占的空间越大、检索越慢。支持 Matryoshka（MRL）的模型可以只取前面一部分维度（2.4.3） |
| 查询、文档前缀 | Qwen3-Embedding、BGE、E5 等训练时给查询和文档加不同前缀，查询时漏加，效果会变差（2.4.3） |
| 部署方式 | API：省事，但按量付费，数据要发出去；本地：数据不出内网，没有调用费，要 GPU 才快 |
| 许可证 | 多数是 Apache-2.0 或 MIT，少数限制商用（如 jina-embeddings-v4） |

本节用两个模型，都通过同一个 OpenAI 兼容的中转站调用（`.env` 里的 `embedding_model_base`）：`Qwen3-VL-Embedding-2B` 和 `jina-embeddings-v4`。创建嵌入模型有两种写法：

1. `init_embeddings("提供方:模型名", **kwargs)`：langchain 1.x 的统一入口（0.3.9 加入），和第 2 章的 `init_chat_model` 类似；
2. 直接实例化集成类，例如 `OpenAIEmbeddings(...)`。

两种写法都加了 `check_embedding_ctx_length=False`，原因见下面的“用 `OpenAIEmbeddings` 调非 OpenAI 模型”。

In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.tools import tool
from langchain.embeddings import init_embeddings

load_dotenv(override=True, verbose=True, dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")



embedding_model = init_embeddings(
    model="openai:Qwen3-VL-Embedding-2B",
    api_key=os.environ["embedding_model_key"],
    base_url=os.environ["embedding_model_base"],
    check_embedding_ctx_length=False,  # 非 OpenAI 模型：直接发原文，不发 token id
    #profile=custom_profile
)

index_config = {
    "embed": embedding_model,
    "dims": 1024,  # 要和模型实际输出的维度一致（2.4.3 的输出是 1024）
    "fields": ["$"]
}

In [ ]:
from langchain_openai import OpenAIEmbeddings
embedding_model2 = OpenAIEmbeddings(
    model="jina-embeddings-v4",
    api_key=os.environ["embedding_model_key"],
    base_url=os.environ["embedding_model_base"],
    check_embedding_ctx_length=False,  # 非 OpenAI 模型：直接发原文，不发 token id，因为不同模型的词表可能不同
)

`index_config` 是第 9 章长期记忆的 store 用的配置，本节没有用到。原来写的 `"dims": 512` 和模型实际输出的 1024 维不一致，已改成 1024（第 9 章说过，只有 PostgresStore 会按 `dims` 建表，维度不对会出错）。

**`init_embeddings` 做了什么**（`langchain/embeddings/base.py`，注释是我加的）：

```python
provider, model_name = _parse_model_string("openai:Qwen3-VL-Embedding-2B")
# 按第一个冒号拆开 → ("openai", "Qwen3-VL-Embedding-2B")
cls = _BUILTIN_PROVIDERS["openai"]
# 查表 → langchain_openai 包里的 OpenAIEmbeddings 类
return cls(model="Qwen3-VL-Embedding-2B", **kwargs)
# api_key、base_url、check_embedding_ctx_length 原样传给这个类
```

所以前缀 `openai:` 的意思是“用 `OpenAIEmbeddings`、按 OpenAI 的接口格式发请求”，不是“OpenAI 的模型”。上面两个单元格创建的是同一个类：

In [ ]:
for name, m in [("embedding_model", embedding_model), ("embedding_model2", embedding_model2)]:
    print(f"{name}: {type(m).__module__}.{type(m).__name__}, model={m.model!r}")

embedding_model: langchain_openai.embeddings.base.OpenAIEmbeddings, model='Qwen3-VL-Embedding-2B'
embedding_model2: langchain_openai.embeddings.base.OpenAIEmbeddings, model='jina-embeddings-v4'


`_BUILTIN_PROVIDERS` 里一共 10 个前缀：`openai`、`azure_openai`、`azure_ai`、`bedrock`、`cohere`、`google_genai`、`google_vertexai`、`huggingface`、`mistralai`、`ollama`。不在表里的（比如 Voyage）会报 `ValueError`，只能直接用它的类。

#### 用 `OpenAIEmbeddings` 调非 OpenAI 模型：`check_embedding_ctx_length`

`OpenAIEmbeddings` 默认 `check_embedding_ctx_length=True`：先用 OpenAI 的分词器 tiktoken 把文本变成 token id，再把 **token id** 发给服务端，而不是原文：

![OpenAIEmbeddings 的请求流程](assets/12-openai-embeddings-request.svg)

用 httpx 的事件钩子把真正发出去的请求体打印出来：

In [ ]:
import httpx
from langchain_openai import OpenAIEmbeddings

def print_body(request):
    print("  请求体:", request.content.decode())

vectors = {}
for flag in [True, False]:
    print(f"check_embedding_ctx_length={flag}")
    m = OpenAIEmbeddings(
        model="Qwen3-VL-Embedding-2B",
        api_key=os.environ["embedding_model_key"],
        base_url=os.environ["embedding_model_base"],
        check_embedding_ctx_length=flag,
        http_client=httpx.Client(event_hooks={"request": [print_body]}),  # 每次发请求前调用 print_body
    )
    vectors[flag] = m.embed_query("你好，很高兴认识你")

print("两种写法得到的向量相同：", vectors[True] == vectors[False])

check_embedding_ctx_length=True
  请求体: {"input":[[57668,53901,3922,17599,230,45736,6708,112,30051,6744,228,57668]],"model":"Qwen3-VL-Embedding-2B","encoding_format":"base64"}


check_embedding_ctx_length=False
  请求体: {"input":["你好，很高兴认识你"],"model":"Qwen3-VL-Embedding-2B","encoding_format":"base64"}


两种写法得到的向量相同： True


* `True`（默认）：`input` 是 `[[57668, 53901, …]]`，这是“你好，很高兴认识你”在 OpenAI `cl100k_base` 词表里的编号。`Qwen3-VL-Embedding-2B` 不是 tiktoken 认识的模型名，源码就退回 `cl100k_base`；
* `False`：`input` 是原文，由服务端用模型自己的分词器处理。

Qwen、jina、bge 等模型的词表和 OpenAI 不同。本笔记用的中转站两种写法结果相同，说明它把 token id 还原成了文字；但很多 OpenAI 兼容服务（vLLM、Ollama 等）不支持 token id 输入，或者会按自己的词表去理解这些编号，得到的向量就错了。`OpenAIEmbeddings` 的源码注释和官方文档都建议：调非 OpenAI 的模型时设 `check_embedding_ctx_length=False`。

> ⚠️ **原笔记的写法有隐患，已修正**：上面两个模型原来都没有设 `check_embedding_ctx_length=False`，已经补上。换成这个中转站之外的服务，原来的写法可能报错或得到错误的向量。这个中转站两种写法的结果一样，所以 2.4.3、2.4.4 保存的输出仍然有效。

### 2.4.3 查询的向量化（embed_query）

> ⚠️ 原标题是“句子的向量化”，按官方文档改成“查询的向量化”，原因见 2.4.1。

In [4]:

text = "你好，很高兴认识你"


embed_docs = embedding_model.embed_query(text)

print(len(embed_docs))

print(embed_docs[:5])

1024
[0.01232147216796875, -0.020965576171875, 0.030303955078125, 0.002002716064453125, 0.0156707763671875]


In [6]:

text = "你好，很高兴认识你"


embed_docs = embedding_model2.embed_query(text)

print(len(embed_docs))

print(embed_docs[:5])

2048
[0.0131248002871871, -0.03309205546975136, 0.004701917991042137, 0.015739163383841515, -0.004711000714451075]


`embed_query` 返回 `list[float]`，长度就是维度：`Qwen3-VL-Embedding-2B` 是 1024，`jina-embeddings-v4` 是 2048。两个模型的向量在不同的空间里，同一句话的两个向量也不能拿来比较，入库和查询必须用同一个模型。

**`OpenAIEmbeddings` 的 `embed_query` 和文档没有区别**。源码只有一行：

```python
def embed_query(self, text: str, **kwargs: Any) -> list[float]:
    return self.embed_documents([text], **kwargs)[0]
```

但有些模型训练时，查询和文档的处理方式不同，查询时要加前缀或传参数：

| 模型 | 查询 | 文档 |
|---|---|---|
| Qwen3-Embedding | `Instruct: {任务描述}\nQuery:{问题}` | 原文 |
| bge-*-zh-v1.5 | 可选：`为这个句子生成表示以用于检索相关文章：` + 问题（模型卡说 v1.5 不加只是略有下降） | 原文 |
| multilingual-e5 | `query: ` + 问题 | `passage: ` + 原文 |
| jina-embeddings-v4 | 检索任务，`prompt_name="query"` | `prompt_name="passage"` |
| Voyage、Cohere 的 API | `input_type` 设为查询 | `input_type` 设为文档 |

* `VoyageAIEmbeddings`、`CohereEmbeddings` 在 `embed_query` 和 `embed_documents` 里各自传对应的 `input_type`，不用自己管；
* 本地用 `HuggingFaceEmbeddings` 时，按官方文档分别设 `encode_kwargs={"prompt": "passage: "}` 和 `query_encode_kwargs={"prompt": "query: "}`；
* 通过 OpenAI 兼容接口调用时，接口里没有这个参数，只能自己把前缀拼进查询文本（先看服务端会不会自动加）。`Qwen3-VL-Embedding-2B` 的模型卡说它默认给所有输入加同一句指令，不区分查询和文档。

#### 维度：`Qwen3-VL-Embedding-2B` 为什么是 1024 维？

模型卡写的是“最多 2048 维，支持 64～2048 之间自定义”，这个中转站默认返回 1024 维。`OpenAIEmbeddings` 的 `dimensions` 参数会原样放进请求体，试试 512 和 2048：

In [ ]:
import numpy as np

text = "你好，很高兴认识你"
for d in [None, 512, 2048]:
    m = OpenAIEmbeddings(
        model="Qwen3-VL-Embedding-2B",
        api_key=os.environ["embedding_model_key"],
        base_url=os.environ["embedding_model_base"],
        check_embedding_ctx_length=False,
        dimensions=d,  # None 表示不传，用服务端的默认值
    )
    v = m.embed_query(text)
    print(f"dimensions={d}: 长度 {len(v)}，向量长度（范数）{np.linalg.norm(v):.4f}，前 3 个数 {v[:3]}")

v_jina = embedding_model2.embed_query(text)
print(f"\njina-embeddings-v4: 长度 {len(v_jina)}，向量长度（范数）{np.linalg.norm(v_jina):.4f}")

dimensions=None: 长度 1024，向量长度（范数）0.7064，前 3 个数 [0.01232147216796875, -0.020965576171875, 0.030303955078125]


dimensions=512: 长度 512，向量长度（范数）0.5223，前 3 个数 [0.01232147216796875, -0.020965576171875, 0.030303955078125]


dimensions=2048: 长度 2048，向量长度（范数）0.9998，前 3 个数 [0.01232147216796875, -0.020965576171875, 0.030303955078125]



jina-embeddings-v4: 长度 2048，向量长度（范数）1.0000


从输出能看出两件事：

1. **短向量是长向量的前一段**：三次的前 3 个数完全一样。`Qwen3-VL-Embedding-2B` 用 Matryoshka 表示学习（MRL）训练，前 k 维单独拿出来也能用，所以要更短的向量时，直接截取前面一段。维度越小，存储越省、检索越快，效果会稍差一点；
2. **截短后没有重新归一化**：2048 维的向量长度约为 1，截成 1024 维只剩约 0.71，512 维约 0.52。jina-embeddings-v4 返回的是归一化的向量（长度 1）。

后一点影响相似度的选择：余弦相似度会除以向量长度，不受影响；点积不再等于余弦相似度。如果向量库按点积（内积）检索，要先把向量归一化。

> ⚠️ **补充**：`OpenAIEmbeddings` 源码注释说 `dimensions` “只有 text-embedding-3 及以后的模型支持”，指的是 OpenAI 自己的模型。调用其他服务时，`dimensions` 只是放进请求体，服务端认不认要看它的文档；不认的可能报错，也可能直接忽略。

### 2.4.4 文档的向量化（embed_documents）
文档的向量化，接收的参数是字符串数组。

In [7]:

# 待嵌入的文本列表
texts = [
    "Hi there!",
    "Oh, hello!",
    "What's your name?",
    "My friends call me World",
    "Hello World!"
]

# 生成嵌入向量
embeded_docs = embedding_model.embed_documents(texts)

for i in range(len(texts)):
    print(f"{texts[i]}:{embeded_docs[i][:3]}",end="\n\n")

Hi there!:[0.03607177734375, -0.0236663818359375, 0.030609130859375]

Oh, hello!:[0.0147552490234375, -0.0271759033203125, 0.040802001953125]

What's your name?:[0.03277587890625, -0.01226043701171875, 0.00859832763671875]

My friends call me World:[-0.0035839080810546875, -0.038238525390625, 0.0179595947265625]

Hello World!:[-0.01491546630859375, -0.006175994873046875, 0.007595062255859375]



In [8]:

# 待嵌入的文本列表
texts = [
    "Hi there!",
    "Oh, hello!",
    "What's your name?",
    "My friends call me World",
    "Hello World!"
]

# 生成嵌入向量
embeded_docs = embedding_model2.embed_documents(texts)

for i in range(len(texts)):
    print(f"{texts[i]}:{embeded_docs[i][:3]}",end="\n\n")

Hi there!:[0.0020421501249074936, -0.026686308905482292, 0.02462201938033104]

Oh, hello!:[0.0018917950801551342, -0.03210397809743881, -0.00561273330822587]

What's your name?:[-0.006894869264215231, -0.010459912940859795, -0.04930686950683594]

My friends call me World:[-0.01910482533276081, -0.025690464302897453, -0.02106555551290512]

Hello World!:[0.0030771896708756685, -0.029493259266018867, -0.01423666626214981]



`embed_documents` 返回 `list[list[float]]`，一条文本一个向量，顺序和输入一致。

`OpenAIEmbeddings` 发请求时还有两点要注意：

* **分批**：它的 `chunk_size` 参数（默认 1000）是**每次请求最多带几条文本**，和 2.3 切分器的 `chunk_size`（每块多长）不是一回事。有的服务限制每次的条数，例如百炼的 `text-embedding-v4` 一次最多 10 条，要设 `chunk_size=10`，否则请求会报错。也可以每次调用时传：`embed_documents(texts, chunk_size=10)`；
* **超长文本**：`check_embedding_ctx_length=True` 时，超过 8191 个 token 的文本被切成几段，分别算向量后按 token 数加权平均（上面的图）。8191 是 OpenAI 模型的上限，对其他模型没有意义。设成 `False` 后由服务端处理，一般是截断或报错。所以切分时块长度要小于所用模型的最大输入长度。

#### 用向量算相似度

`embed_query` 得到问题的向量，`embed_documents` 得到文档的向量，再按 2.4.1 的余弦相似度比较，就能看出哪段文本和问题的意思最接近（2.5 的向量库在大量向量上做的就是这件事）：

In [ ]:
def cosine_similarity(vec1, vec2):  # 官方文档给的写法
    dot = np.dot(vec1, vec2)
    return dot / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

query = "宠物猫平时吃什么？"
doc_texts = [
    "猫喜欢吃鱼，也爱吃鸡胸肉",
    "狗每天都需要出门散步",
    "今天 A 股收盘下跌 2%",
    "Cats love to eat fish.",
]

query_vec = embedding_model.embed_query(query)
doc_vecs = embedding_model.embed_documents(doc_texts)

scores = [cosine_similarity(query_vec, v) for v in doc_vecs]
for score, doc in sorted(zip(scores, doc_texts), reverse=True):
    print(f"{score:.3f}  {doc}")

0.719  猫喜欢吃鱼，也爱吃鸡胸肉
0.654  Cats love to eat fish.
0.496  狗每天都需要出门散步
0.286  今天 A 股收盘下跌 2%


排在前面的两条都和“猫吃什么”有关。英文的“Cats love to eat fish.”和问题没有一个字相同，分数仍然比“狗”“A 股”高：多语言模型把不同语言里意思相同的句子也映射到相近的位置。

csv 文件的向量化

In [ ]:
from langchain_community.document_loaders import CSVLoader


# 情况1：
loader = CSVLoader("load/02-load.csv", encoding="utf-8")
docs = loader.load_and_split()#默认切分使用递归切分的方法

print(len(docs))
for doc in docs:
    print(doc)
# 存放的是每一个chrunk的embedding。

# texts = [doc.page_content for doc in docs]

# embeded_docs = embedding_model2.embed_documents(texts)

# print(len(embeded_docs))

# for i in range(len(texts)):
#     print(f"{texts[i]}:\n{embeded_docs[i][:3]}",end="\n\n")

4
page_content='id: 1
title: Introduction to Python
content: Python is a popular programming language.
author: John Doe' metadata={'source': 'load/02-load.csv', 'row': 0}
page_content='id: 2
title: Data Science Basics
content: Data science involves statistics and machine learning.
author: Jane Smith' metadata={'source': 'load/02-load.csv', 'row': 1}
page_content='id: 3
title: Web Development
content: HTML, CSS and JavaScript are core web technologies.
author: Mike Johnson' metadata={'source': 'load/02-load.csv', 'row': 2}
page_content='id: 4
title: Artificial Intelligence
content: AI is transforming many industries.
author: Sarah Williams' metadata={'source': 'load/02-load.csv', 'row': 3}


`load_and_split()` 没传切分器时，用默认参数的 `RecursiveCharacterTextSplitter()`（`chunk_size=4000`、`chunk_overlap=200`）。这 4 行都很短，所以每行还是一块。

> ⚠️ **补充**：`BaseLoader.load_and_split` 的源码注释写着 “It should be considered to be deprecated!”（应视为已弃用）。按 2.3 的写法，先 `loader.load()`，再用自己配置的切分器 `split_documents(docs)`。

把每块的 `page_content` 交给 `embed_documents`（上面单元格注释掉的那几行）：

In [ ]:
texts = [doc.page_content for doc in docs]

embeded_docs = embedding_model2.embed_documents(texts)

print(len(embeded_docs), len(embeded_docs[0]))

for i in range(len(texts)):
    print(f"{texts[i]}:\n{embeded_docs[i][:3]}", end="\n\n")

4 2048
id: 1
title: Introduction to Python
content: Python is a popular programming language.
author: John Doe:
[-0.005929863546043634, -0.00954402331262827, 0.010283292271196842]

id: 2
title: Data Science Basics
content: Data science involves statistics and machine learning.
author: Jane Smith:
[0.029238255694508553, 0.017072144895792007, 0.03654791787266731]

id: 3
title: Web Development
content: HTML, CSS and JavaScript are core web technologies.
author: Mike Johnson:
[0.03277948498725891, -0.019007213413715363, 0.016851378604769707]

id: 4
title: Artificial Intelligence
content: AI is transforming many industries.
author: Sarah Williams:
[0.012075033970177174, 0.012089909054338932, 0.02726958692073822]



4 块得到 4 个 2048 维的向量。注意两点：

* 只有 `page_content` 被向量化，`metadata`（`source`、`row`）不参与计算，存进向量库时和向量放在一起（2.5）；
* `CSVLoader` 把每一列写成“列名: 值”，所以 `id: 1`、`author: John Doe` 也被算进了向量。只想让 `title`、`content` 参与语义检索，可以设 `metadata_columns=["id", "author"]`，把这两列移到 `metadata` 里（2.2.2 `CSVLoader` 的参数）。

### 2.4.5 嵌入缓存 CacheBackedEmbeddings

> 官方文档的 Embedding models 页有这一节，原笔记没有，这里补上。

同一批文档重复入库时（重跑 notebook、增量更新知识库），每次都会重新调用嵌入接口、重新付费。`CacheBackedEmbeddings` 包在嵌入模型外面，把“文本 → 向量”存进一个键值存储（`ByteStore`），算过的文本直接从缓存取。

> ⚠️ **与旧教程不同**：旧教程从 `langchain.embeddings` 导入它。langchain 1.0 把它移到了 `langchain-classic`，现在写 `from langchain_classic.embeddings import CacheBackedEmbeddings`（官方文档也是这么写的）。

一般用 `CacheBackedEmbeddings.from_bytes_store(...)` 创建：

| 参数 | 默认值 | 含义 |
|---|---|---|
| `underlying_embedder` | 必填 | 真正计算向量的嵌入模型 |
| `document_embedding_cache` | 必填 | 存文档向量的 `ByteStore`，例如 `LocalFileStore("./cache/")`（存成本地文件）、`InMemoryByteStore()`（存在内存里） |
| `batch_size` | `None` | 每算多少条写一次缓存，`None` 表示全部算完再写 |
| `namespace` | `""` | 缓存键的前缀。**一定要设**，一般用模型名：不设的话，换了模型还会取到旧模型的向量 |
| `query_embedding_cache` | `False` | 默认**不缓存查询**；传 `True` 表示和文档共用一个存储，也可以单独传一个 `ByteStore` |
| `key_encoder` | `"sha1"` | 文本变成缓存键用的哈希算法。用 `"sha1"` 会发出一次警告（可能碰撞），可以改成 `"sha256"` 或 `"blake2b"` |

`embed_documents` 的过程（`langchain_classic/embeddings/cache.py`）：先用 `mget` 按文本批量查缓存，找出没查到的那些，只把它们交给 `underlying_embedder.embed_documents` 计算，再用 `mset` 写回缓存，最后按原顺序返回。

In [ ]:
import time
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_core.stores import InMemoryByteStore

store = InMemoryByteStore()
cached_embedder = CacheBackedEmbeddings.from_bytes_store(
    embedding_model,
    store,
    namespace=embedding_model.model,  # 用模型名做前缀
    key_encoder="sha256",
)

texts = ["Hi there!", "Oh, hello!", "What's your name?", "My friends call me World", "Hello World!"]
for i in range(2):
    tic = time.time()
    vectors = cached_embedder.embed_documents(texts)
    print(f"第 {i + 1} 次：{time.time() - tic:.3f} 秒，缓存里有 {len(list(store.yield_keys()))} 条")

print("一个缓存键：", next(store.yield_keys()))
print("和直接调用的结果相同：", vectors[0] == embedding_model.embed_documents(texts[:1])[0])

第 1 次：0.379 秒，缓存里有 5 条
第 2 次：0.001 秒，缓存里有 5 条
一个缓存键： Qwen3-VL-Embedding-2Bd451d2a79e0a1f87270b313ca7d9e589359cb3d2aa906594529dcbc0535fd0f3


和直接调用的结果相同： True


第 1 次调用接口并写入 5 条缓存，第 2 次全部从缓存读取，几乎不花时间。缓存键是“命名空间 + 文本的哈希”。

`InMemoryByteStore` 在进程结束后就没了，只适合演示；要跨次运行保留，用 `LocalFileStore` 或数据库类的存储（官方文档的 stores 集成页）。

### 2.4.6 面试常见问题

**1. `embed_query` 和 `embed_documents` 有什么区别？为什么要分成两个方法？**

输入输出不同：一条字符串得到一个向量，多条字符串得到多个向量。分开是为了**非对称检索**：问题短、文档长，很多模型（Qwen3-Embedding、BGE、E5，Voyage 和 Cohere 的 API）训练时给查询和文档加不同的前缀或类型。分成两个方法，集成类就能在查询这边自动加上。对 `OpenAIEmbeddings` 来说两者没有区别（2.4.3）。

**2. 为什么入库和查询必须用同一个嵌入模型？想换模型怎么办？**

每个模型有自己的向量空间。维度不同，向量根本没法计算；维度相同（比如都是 1024），每一维的含义也不同，算出来的相似度没有意义。同一家的新旧版本也不通用，例如 Google 说明 `gemini-embedding-2` 的向量和 `gemini-embedding-001` 不兼容。换模型就要用新模型把全部文档重新向量化、重建索引，所以入库时最好记下用的模型和维度。

**3. 怎么选嵌入模型？**

看效果（MTEB 榜单加自己的数据测试）、语言、最大输入长度、维度、部署方式和成本、许可证，见 2.4.2 的表。

**4. 向量维度越高越好吗？什么是 Matryoshka（MRL）？**

维度高通常效果更好，但存储和计算开销跟着线性增长：1024 维的 float32 向量一条 4 KB，100 万条约 4 GB。MRL 训练让前 k 维单独也能用，需要时直接截取前 k 维，用一点效果换存储和速度（2.4.3 的验证：前几个数相同）。截短后要重新归一化。

**5. 余弦相似度、点积、欧氏距离怎么选？**

余弦相似度只看方向，点积还受长度影响，欧氏距离看两点之间的距离。向量归一化后三者的排序一样（2.4.1）。一般按模型文档的推荐，多数用余弦相似度；不确定向量是否归一化时，用余弦相似度，或者先归一化再用点积（2.4.3 的中转站就返回了没有归一化的向量）。

**6. 文本超过嵌入模型的最大输入长度会怎样？块大小怎么定？**

多数模型和服务会截断，超出的部分不进入向量，检索时就找不到这部分内容；有的服务直接报错；`OpenAIEmbeddings` 默认按 8191 个 token 切段后加权平均（2.4.2 的图）。所以切分时，块长度（按 token 算）要小于模型的最大输入。块也不宜太长：一个向量要概括的内容越多，每个细节在向量里的分量越小，检索就越不准。

**7. 怎么降低嵌入的成本、提高速度？**

* 批量调用：`embed_documents` 一次请求带多条；
* 缓存：`CacheBackedEmbeddings`，算过的文本不再算（2.4.5）；
* 异步并发：`aembed_documents`；
* 降维：支持 MRL 的模型取较短的向量；
* 量大时在本地部署开源模型（TEI、vLLM、Ollama），没有按量计费。

**8. 只用向量检索有什么不足？**

稠密向量擅长找“意思相近”的内容，不擅长精确匹配：产品型号、人名、代码里的标识符、数字等。常见的做法是把向量检索和关键词检索（BM25）或稀疏向量（bge-m3 能同时输出）结合起来，叫混合检索（hybrid retrieval）。